In [6]:
import os
import time
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

import igraph as ig
import leidenalg as la
from sklearn.metrics import adjusted_rand_score
from itertools import combinations
import pickle

In [2]:
from IPython.display import Image, display

out_dir = "./data/senepy_denovo_signatures_code/leiden_grid_search_output"

with open(os.path.join(out_dir, "keys_list.txt"), "r") as f:
    lines = f.readlines()

# удаляем символ переноса строки
lines = [line.strip() for line in lines]


In [ ]:
for tc in lines:
    meta_df = pd.read_csv(os.path.join(out_dir, f"{tc}.metadata.csv"), index_col=0)
    la_gridsearch_df = pd.read_csv(
        os.path.join(out_dir, f"{tc}.la_gridsearch.csv"), index_col=0
    )

    display(tc, meta_df, la_gridsearch_df)
    display(Image(filename=os.path.join(out_dir, f"{tc}.clusters_grid.png")))

In [3]:
res_dict = {
    "Colon.CD4T": 0.1,  # wasted
    "Colon.CD8T": 0.1,  # wasted
    "Colon.Crypt stem cell": 0.25,  # 3
    "Colon.Endothelial cell": 0.5,  # 2
    "Colon.Enterocyte": 0.3,  # 2
    "Colon.Fibroblast": 0.3,
    "Colon.Goblet cell": 0.35,  # а так оно там все отвалится
    "Colon.Macrophage": 0.35,  # 1
    "Colon.Mast cell": 0.35,  # 1
    "Colon.Plasma cell": 0.35,  # два кластера. Мелкий граф, второй кластер отвалится
    "Colon.Transit amplifying cell": 0.35,
    "Colon.Tuft cell": 0.2,
    "Colon.gdT": 0.2,
    "Heart_Cell.Atrial Cardiomyocyte": 0.25,  # один кластер
    "Heart_Cell.Endothelial cell": 0.2,  # 1
    "Heart_Cell.Mural cell": 0.35,  # 3
    "Heart_Cell.Ventricular Cardiomyocyte": 0.35,  # 1 big
    "Heart_Nuclei.Adipocyte": 0.4,  # 2
    "Heart_Nuclei.Atrial Cardiomyocyte": 0.3,  # 3
    "Heart_Nuclei.CD4T": 0.5,  # 2
    "Heart_Nuclei.CD8T": 0.1,
    "Heart_Nuclei.Endothelial cell": 0.2,  # 1
    "Heart_Nuclei.Fibroblast": 0.35,  # 2
    "Heart_Nuclei.Macrophage": 0.7,  # 3
    "Heart_Nuclei.Mono+mac": 0.35,  # 2
    "Heart_Nuclei.Monocyte": 0.75,  # 3
    "Heart_Nuclei.Mural cell": 0.35,  # 2 or 1
    "Heart_Nuclei.Neural cell": 0.35,  # 1
    "Heart_Nuclei.Ventricular Cardiomyocyte": 0.25,  # 2
    "Ileum.B cell": 0.15,  # хз, это планета)) ожидаю 2 кластера
    "Ileum.CD4T": 0.2,  # 3, -//-
    "Ileum.CD8T": 0.2,  # 2, -//-
    "Ileum.Crypt stem cell": 0.15,  # 1
    "Ileum.Endothelial cell": 0.2,  # 1
    "Ileum.Enterocyte": 0.25,  # 3
    "Ileum.Enteroendocrine cell": 0.05,  # 1
    "Ileum.Fibroblast": 0.15,  # 1
    "Ileum.Goblet cell": 0.15,  # 1
    "Ileum.Paneth cell": 0.15,  # 1
    "Ileum.Plasma cell": 0.05,  # 1
    "Ileum.gdT": 0.05,  # 1
    "Kidney.CD4T": 0.2,  # 1
    "Kidney.CD8T": 0.35,  # 1!!!
    "Kidney.Endothelial cell": 0.7,  # 3
    "Kidney.Epithelial cell": 0.35,  # 2
    "Kidney.Fibroblast": 0.2,  # 2 !!!
    "Kidney.Mono+mac": 0.35,  # 2
    "Kidney.NK": 0.25,  # 1 big
    "Lung.Adventitial cell": 0.35,  # 2
    "Lung.Alveolar adventitial fibroblast": 0.25,  # 3
    "Lung.Basal cell": 0.25,  # 1
    "Lung.Basophil": 0.1,  # 1
    "Lung.Bronchial smooth muscle cell": 0.4,  # 2
    "Lung.CD4T": 0.1,  # 1
    "Lung.CD8T": 0.1,  # 1
    "Lung.Capillary endothelial cell": 0.4,  # 2
    "Lung.Club cell": 0.2,  # 1
    "Lung.Endothelial cell of artery": 0.15,  # 2
    "Lung.Endothelial cell of lymphatic vessel": 0.65,  # 2
    "Lung.Macrophage": 0.1,  # 1
    "Lung.Monocyte": 0.25,  # 2
    "Lung.Multiciliated epithelial cell": 0.4,  # 2
    "Lung.NKT": 0.1,  # wasted
    "Lung.Pericyte": 0.1,  # wasted
    "Lung.Plasma cell": 0.1,  # 1
    "Lung.Pulmonary alveolar type 1 cell": 0.6,  # 3
    "Lung.Pulmonary alveolar type 2 cell": 0.7,  # 4
    "Lung.Respiratory tract goblet cell": 0.35,  # 2
    "Lung.Vein endothelial cell": 0.15,  # 2
    "Skin.CD4T": 0.2,  # 1 big
    "Skin.CD8T": 0.25,  # 1 big
    "Skin.DC": 0.25,  # 1 big
    "Skin.Differentiated_KC": 0.2,  # 1 or 2
    "Skin.Fibroblast": 0.35,  # 1
    "Skin.LE": 0.25,  # 1
    "Skin.Macrophage": 0.2,  # 1,
    "Skin.Mast cell": 0.25,  # 1
    "Skin.Melanocyte": 0.2,  # 1
    "Skin.NK": 0.2,  # 1
    "Skin.Pericyte": 0.25,  # 1
    "Skin.Proliferating_KC": 0.05,  # wasted
    "Skin.Treg": 0.35,  # 1
    "Skin.Undifferentiated_KC": 0.05,  # 1
    "Skin.VE": 0.4,  # 2
}

In [7]:
file_path = "./data/senepy_denovo_signatures_code/tc_resolution_dict.pkl"

with open(file_path, "wb") as file:
    pickle.dump(res_dict, file)